# Experiment: Mininet Actual Experiment

실제 MP4 비디오를 입력으로 사용하고, WSL2 Ubuntu의 Mininet에서 실제 TCP/UDP 전송을 수행한 뒤 `late_frame_ratio`와 `late_frame_count / frame_count`를 비교한다.


In [1]:
# cell 1 : 사용자 실험 파라미터 설정
# ──────────────────────────────────────────────
# 이 셀에서만 실험 조건을 수정하세요.
# ──────────────────────────────────────────────

# 실험 대상 비디오 목록
VIDEO_NAMES = [
    "archive_popeye_512kb",
    "echo_mediaelement",
    "w3c_movie_300",
]

# 비교할 전송 정책 목록
POLICY_NAMES = ["heuristic_frame_aware", "frame_action_single_path"]

# 대역폭 조건 (Mbps) — 원하는 값을 자유롭게 추가/제거/변경 가능
# BANDWIDTH_VALUES_MBPS = [3, 2, 1]
BANDWIDTH_VALUES_MBPS = [30, 10, 5, 3, 1]

# 조건별 반복 횟수
REPEAT_COUNT = 3

# 네트워크 환경 파라미터
ROUND_TRIP_TIME_MS = 10
LOSS_RATE = 0.0

# 재생 버퍼 (ms) — 프레임 데드라인 계산에 사용
PLAYBACK_BUFFER_MS = 50.0


In [2]:
# cell 2 : 실험 인프라 (import, 유틸리티 함수, 경로)
from __future__ import annotations

from pathlib import Path
import json
import shlex
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

REPO_ROOT = Path(r"C:/git/network")
OUTPUT_ROOT = REPO_ROOT / "output" / "mininet_actual_experiment"
WSL_PYTHON = "python3"
WSL_INSTALL_COMMAND = "sudo apt update && sudo apt install -y ffmpeg mininet python3-av"

def to_wsl_path(path: Path) -> str:
    drive = path.drive.rstrip(":").lower()
    suffix = path.as_posix().split(":", 1)[-1]
    return f"/mnt/{drive}{suffix}"

def run_wsl_command(command: str, *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print(command)
    completed = subprocess.run(
        ["wsl", "-e", "bash", "-lc", command],
        check=False,
        text=True,
        capture_output=True,
    )
    if check and completed.returncode != 0:
        raise RuntimeError(
            "WSL command failed.\n"
            f"exit_code: {completed.returncode}\n\n"
            f"stdout:\n{completed.stdout}\n\n"
            f"stderr:\n{completed.stderr}"
        )
    return completed

def run_wsl_python(script_relative_path: str, *args: str, use_sudo: bool = False) -> subprocess.CompletedProcess[str]:
    command_parts = [WSL_PYTHON, script_relative_path, *args]
    if use_sudo:
        command_parts = ["sudo", "-n", *command_parts]
    command = (
        f"cd {shlex.quote(to_wsl_path(REPO_ROOT))} && "
        + " ".join(shlex.quote(part) for part in command_parts)
    )
    return run_wsl_command(command)

def ensure_wsl_sudo_ready() -> None:
    command = f"cd {shlex.quote(to_wsl_path(REPO_ROOT))} && sudo -n true"
    completed = run_wsl_command(command, check=False)
    if completed.returncode != 0:
        raise RuntimeError(
            "WSL sudo credentials are not cached. "
            'PowerShell에서 `wsl -e bash -lc "sudo -v"`를 먼저 실행한 뒤 다시 시도하세요.'
        )

REPO_ROOT


WindowsPath('C:/git/network')

In [3]:
# cell 3 : WSL 환경 점검
preflight_result = run_wsl_python("scripts/check_actual_experiment_env.py")
print(preflight_result.stdout)
preflight_status = json.loads(preflight_result.stdout)
if not preflight_status["ready"]:
    missing_dependencies = [
        name for name in ("python3", "ffprobe", "mn", "pyav")
        if not preflight_status[name]
    ]
    raise RuntimeError(
        "WSL2 Ubuntu 실험 환경이 준비되지 않았습니다.\n"
        f"누락 의존성: {missing_dependencies}\n"
        f"설치 명령: {WSL_INSTALL_COMMAND}"
    )


cd /mnt/c/git/network && python3 scripts/check_actual_experiment_env.py
{
  "platform": "Linux-6.6.114.1-microsoft-standard-WSL2-x86_64-with-glibc2.39",
  "python3": true,
  "ffprobe": true,
  "mn": true,
  "pyav": true,
  "ready": true
}



In [4]:
# cell 4 : trace CSV 준비
for video_name in VIDEO_NAMES:
    video_path = REPO_ROOT / "dataset" / "videos" / f"{video_name}.mp4"
    output_csv_path = REPO_ROOT / "dataset" / "traces" / f"{video_name}_trace.csv"
    result = run_wsl_python(
        "scripts/video_trace_prepare.py",
        "--video-path", to_wsl_path(video_path),
        "--output-csv", to_wsl_path(output_csv_path),
        "--playback-buffer-ms", str(PLAYBACK_BUFFER_MS),
    )
    print(result.stdout)


cd /mnt/c/git/network && python3 scripts/video_trace_prepare.py --video-path /mnt/c/git/network/dataset/videos/archive_popeye_512kb.mp4 --output-csv /mnt/c/git/network/dataset/traces/archive_popeye_512kb_trace.csv --playback-buffer-ms 50.0
{
  "video_path": "/mnt/c/git/network/dataset/videos/archive_popeye_512kb.mp4",
  "output_csv_path": "/mnt/c/git/network/dataset/traces/archive_popeye_512kb_trace.csv",
  "frame_count": 11098,
  "gop_count": 925,
  "total_payload_bytes": 23786543
}

cd /mnt/c/git/network && python3 scripts/video_trace_prepare.py --video-path /mnt/c/git/network/dataset/videos/echo_mediaelement.mp4 --output-csv /mnt/c/git/network/dataset/traces/echo_mediaelement_trace.csv --playback-buffer-ms 50.0
{
  "video_path": "/mnt/c/git/network/dataset/videos/echo_mediaelement.mp4",
  "output_csv_path": "/mnt/c/git/network/dataset/traces/echo_mediaelement_trace.csv",
  "frame_count": 1338,
  "gop_count": 14,
  "total_payload_bytes": 4560751
}

cd /mnt/c/git/network && python3 sc

In [ ]:
# cell 5 : Mininet 실제 실험 실행
ensure_wsl_sudo_ready()

for video_name in VIDEO_NAMES:
    for policy_name in POLICY_NAMES:
        for bandwidth_mbps in BANDWIDTH_VALUES_MBPS:
            result = run_wsl_python(
                "scripts/mininet_actual_experiment.py",
                "--video-name", video_name,
                "--policy-name", policy_name,
                "--bandwidth-mbps", str(bandwidth_mbps),
                "--round-trip-time-ms", str(ROUND_TRIP_TIME_MS),
                "--loss-rate", str(LOSS_RATE),
                "--playback-buffer-ms", str(PLAYBACK_BUFFER_MS),
                "--repeat-count", str(REPEAT_COUNT),
                "--python-executable", WSL_PYTHON,
                use_sudo=True,
            )
            print(result.stdout)


cd /mnt/c/git/network && sudo -n true
cd /mnt/c/git/network && sudo -n python3 scripts/mininet_actual_experiment.py --video-name archive_popeye_512kb --policy-name heuristic_frame_aware --bandwidth-mbps 30 --round-trip-time-ms 10 --loss-rate 0.0 --playback-buffer-ms 50.0 --repeat-count 3 --python-executable python3
{
  "video_name": "archive_popeye_512kb",
  "policy_name": "heuristic_frame_aware",
  "bandwidth_mbps": 30.0,
  "repeat_count": 3,
  "output_directory": "/mnt/c/git/network/output/mininet_actual_experiment/archive_popeye_512kb/heuristic_frame_aware/30mbps",
  "summary_csv": "/mnt/c/git/network/output/mininet_actual_experiment/archive_popeye_512kb/heuristic_frame_aware/30mbps/summary.csv"
}

cd /mnt/c/git/network && sudo -n python3 scripts/mininet_actual_experiment.py --video-name archive_popeye_512kb --policy-name heuristic_frame_aware --bandwidth-mbps 10 --round-trip-time-ms 10 --loss-rate 0.0 --playback-buffer-ms 50.0 --repeat-count 3 --python-executable python3
{
  "video

In [ ]:
# cell 6 : 결과 표 정리
summary_frames = []
for video_name in VIDEO_NAMES:
    for policy_name in POLICY_NAMES:
        for bandwidth_mbps in BANDWIDTH_VALUES_MBPS:
            summary_csv_path = OUTPUT_ROOT / video_name / policy_name / f"{bandwidth_mbps}mbps" / "summary.csv"
            summary_frames.append(pd.read_csv(summary_csv_path))

summary_dataframe = pd.concat(summary_frames, ignore_index=True)
summary_dataframe["late_frame_ratio_text"] = summary_dataframe["late_frame_ratio"].map(lambda value: f"{value:.8f}")
summary_dataframe["late_frame_display"] = (
    summary_dataframe["late_frame_ratio_text"]
    + " ("
    + summary_dataframe["late_frame_fraction"]
    + ")"
)
summary_dataframe[[
    "video_name",
    "policy_name",
    "bandwidth_mbps",
    "late_frame_display",
    "late_frames_per_1000",
    "dropped_frame_count",
    "mean_deadline_miss_ms_on_late",
    "max_deadline_miss_ms",
]]


In [ ]:
# cell 7 : 결과 그래프
figure, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5), constrained_layout=True)

for video_name in VIDEO_NAMES:
    video_frame = summary_dataframe[summary_dataframe["video_name"] == video_name].sort_values("bandwidth_mbps", ascending=False)
    for policy_name in POLICY_NAMES:
        policy_frame = video_frame[video_frame["policy_name"] == policy_name]
        label = f"{video_name} / {policy_name}"
        axes[0].plot(policy_frame["bandwidth_mbps"], policy_frame["late_frame_count"], marker="o", label=label)
        axes[1].plot(policy_frame["bandwidth_mbps"], policy_frame["late_frame_ratio"], marker="o", label=label)

axes[0].set_title("Late Frame Count")
axes[0].set_xlabel("Bandwidth (Mbps)")
axes[0].set_ylabel("Late frame count")
axes[0].invert_xaxis()

axes[1].set_title("Late Frame Ratio")
axes[1].set_xlabel("Bandwidth (Mbps)")
axes[1].set_ylabel("Late frame ratio")
axes[1].invert_xaxis()
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.8f}"))

handles, labels = axes[1].get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", ncol=2)
plt.show()
